# Microcinética: rodar

Entrega as energias, roda a cinética com Monte Carlo nas quatro condições padrão da
tese e grava a rodada para o `validate.ipynb` conferir.

**O que este notebook faz**

1. lê o arquivo de energias (uma linha por espécie, em hartree);
2. monta o perfil de cada reação e as quatro barreiras;
3. roda `n_samples` amostras com energias perturbadas, em cada condição;
4. grava `aggregate_metrics.csv` (no formato do ESI) e um `run_options.json` por condição.

**Antes de rodar:** confira a célula 1 e faça o piloto da célula 3. A rodada completa
da biblioteca é cara (a célula 3 imprime a estimativa).



## 1. Decisões

In [ ]:
from pathlib import Path
import sys, time

import pandas as pd

# O notebook roda de dentro de jupyter/microkinetics/
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from src import config, inputs, mechanism, runner

ROOT = HERE.parents[0]          # a raiz do repositório da tese
ESI = ROOT / "tables"   # as tabelas do material suplementar

# ---- o que rodar ----------------------------------------------------------
# Arquivo de energias do usuário. O formato está no imports/README.md.
ENERGIES_FILE = HERE / "imports/example-energies.csv"

# As energias do arquivo JÁ estão no estado padrão de 1 mol/L?
#   False -> sim (é o caso de quem parte do tables/ch2.libraryBestConformers.csv)
#   True  -> não; o programa soma a correção de 1 atm para 1 mol/L (1,894 kcal/mol por espécie)
ENERGIES_AT_STANDARD_STATE = True

# Reações a rodar. None = todas as que estiverem no arquivo.
# Para um teste: ["im1-ROH3", "im2-ROH4"]
REACTIONS = ["im1-ROH3", "im0-ROH28", "im2-ROH4"]

CONDITIONS = ("A1", "A2", "B1", "B2")
N_SAMPLES = 1000        # o valor da tese
SIGMA = 3.0             # kcal/mol: 3 no protocolo composto, 2 nas reações otimizadas em DFT
RANDOM_SEED = 42        # a semente da tese
N_JOBS = -1             # -1 = todos os núcleos

MODE = runner.MODE_FULL  # "full" é o da tese; "fast" inclui a linha de base nas amostras
SAVE_SAMPLES = True      # grava <reação>_samples.csv (leve)
SAVE_TRAJECTORIES = False  # as trajetórias completas são ~28 MB por reação; deixe desligado

OUTPUT_DIR = HERE / "exports/rodada-exemplo"

print(f"arquivo de energias: {ENERGIES_FILE}")
print(f"saída:               {OUTPUT_DIR}")

## 2. Ler as energias e montar os perfis

Se algo estiver errado com o arquivo, o erro aparece aqui, antes de qualquer simulação.

In [ ]:
energies = mechanism.read_energies(
    ENERGIES_FILE,
    standard_state=not ENERGIES_AT_STANDARD_STATE)

if REACTIONS is not None:
    faltando = [r for r in REACTIONS if r not in energies]
    if faltando:
        raise ValueError(f"reações pedidas que não estão no arquivo: {faltando}")
    energies = {r: energies[r] for r in REACTIONS}

profiles, barriers, skipped = runner.profiles_from_energies(energies)

print("\nBarreiras (kcal/mol):")
display(pd.DataFrame(barriers).T.round(2))

## 3. Piloto: quanto custa a rodada

Roda uma reação em uma condição e mede o tempo. É o número que decide se a rodada
completa cabe no seu tempo — a biblioteca inteira são 136 reações × 4 condições.

In [ ]:
primeira = next(iter(profiles))
t0 = time.time()
_, _, _ = runner.run_reaction(profiles[primeira], runner.settings_for(CONDITIONS[0]),
                              sigma=SIGMA, n_samples=200, random_seed=RANDOM_SEED)
por_amostra = (time.time() - t0) / 201

estimativa = runner.estimate_runtime(len(profiles), CONDITIONS, N_SAMPLES, N_JOBS)
print(f"\n{por_amostra*1000:.0f} ms por amostra (medido agora)")
print(f"rodada completa: {len(profiles)} reações x {len(CONDITIONS)} condições x {N_SAMPLES} amostras")
print(f"estimativa: {estimativa:.0f} min")
print("\nSe for muito, reduza N_SAMPLES (o modo rápido da célula 1 dá um p_MAP")
print("estável com poucas amostras) ou rode um subconjunto de REACTIONS.")

## 4. Rodar

In [ ]:
resultado = runner.run_conditions(
    profiles,
    conditions=CONDITIONS,
    output_dir=OUTPUT_DIR,
    sigma=SIGMA,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
    n_jobs=N_JOBS,
    mode=MODE,
    save_samples=SAVE_SAMPLES,
    save_trajectories=SAVE_TRAJECTORIES)

## 5. Ver o resultado

O p_MAP é a fração das amostras em que o MAP é o majoritário — o lado do corte de 0,5.

In [ ]:
resumo = pd.concat(
    [frame[["p_map", "map_median", "bis_median", "bispyr_median", "n_timed_out"]]
     .assign(condition=condition)
     for condition, frame in resultado.items()])

display(resumo.round(3).head(20))
for condition, frame in resultado.items():
    descartadas = int(frame["n_timed_out"].sum())
    print(f"{condition}: {len(frame)} reações, p_MAP médio {frame['p_map'].mean():.3f}"
          + (f", {descartadas} amostras descartadas por tempo" if descartadas else ""))

print(f"\nrodada gravada em {OUTPUT_DIR}")
print("Agora abra o validate.ipynb e aponte para esta pasta.")